# TruthLens Audio -- v5 notebook

v5 tests one thing: the same combined data v4 used (v2 base + adarshsingh_adf + DEEP-VOICE +
oversampled realistic voice-changing data), but fine-tuned gently from v2's weights (v3's
proven recipe: low lr, few epochs) instead of trained from scratch (v4's recipe, which
collapsed to predicting "ai_generated" for almost every held-out file despite good val F1).

v4 is not retrained here -- it's already trained and checkpointed on Drive, so this notebook
just loads its best checkpoint directly for the final comparison.

Run cells top to bottom. The "Upload realistic eval dataset" cell needs you to manually pick
`audio_eval_dataset.zip` when prompted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

PROJECT_DIR = '/content/drive/MyDrive/TruthLens_Audio'
DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

LABELS = ['genuine', 'ai_generated', 'deepfake']
LABEL_TO_IDX = {l: i for i, l in enumerate(LABELS)}

export_dir = f'{PROJECT_DIR}/export_v2'
export_dir_v3 = f'{PROJECT_DIR}/export_v3'

RUN_NAME_V4 = 'v4'
CKPT_DIR_V4 = f'{PROJECT_DIR}/checkpoints_{RUN_NAME_V4}'
LOG_DIR_V4 = f'{PROJECT_DIR}/logs_{RUN_NAME_V4}'
MANIFEST_DIR_V4 = f'{PROJECT_DIR}/manifests_{RUN_NAME_V4}'
for d in [CKPT_DIR_V4, LOG_DIR_V4, MANIFEST_DIR_V4]:
    os.makedirs(d, exist_ok=True)

print('Setup complete.')


In [ ]:
!pip install -q transformers accelerate soundfile librosa scikit-learn kaggle pandas tqdm av


In [ ]:
import os, shutil
from pathlib import Path

# Safety net only -- restores below should already provide everything, this just covers the
# case where a restore silently comes up short and a real Kaggle download is needed.
kaggle_cred_drive = Path(f'{PROJECT_DIR}/kaggle.json')

if not kaggle_cred_drive.exists():
    print('No cached Kaggle credential found in Drive -- please upload one now.')
    print('(Kaggle -> Settings -> API -> Create New Token)')
    from google.colab import files
    uploaded = files.upload()
    assert uploaded, 'Expected a file to be uploaded'
    uploaded_name = next(iter(uploaded))
    shutil.copy(uploaded_name, kaggle_cred_drive)
    print('Cached to Drive for next time.')

raw = kaggle_cred_drive.read_text().strip()

if raw.startswith('KGAT_'):
    os.environ['KAGGLE_API_TOKEN'] = raw
    legacy_path = Path(os.path.expanduser('~/.kaggle/kaggle.json'))
    if legacy_path.exists():
        legacy_path.unlink()
    print('Detected new-style KGAT_ token -> set KAGGLE_API_TOKEN env var.')
else:
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    shutil.copy(kaggle_cred_drive, os.path.expanduser('~/.kaggle/kaggle.json'))
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
    print('Detected classic kaggle.json -> copied to ~/.kaggle/kaggle.json.')

print('Kaggle credentials ready.')


## Restore raw audio data from Drive backups

In [ ]:
import subprocess

backup_path = f'{PROJECT_DIR}/data_backup.tar.gz'
assert os.path.exists(backup_path), f'No backup found at {backup_path} -- nothing to restore.'

subprocess.run(['cp', backup_path, '/content/data_backup.tar.gz'], check=True)
subprocess.run(['tar', '-xzf', '/content/data_backup.tar.gz', '-C', '/content'], check=True)
print('Restored v2 raw audio data (ASVspoof + FakeAVCeleb + WaveFake) from Drive backup.')


In [ ]:
import subprocess, os

def kaggle_download(dataset_ref, dest_subdir):
    dest = f'{DATA_DIR}/{dest_subdir}'
    if os.path.exists(dest) and os.listdir(dest):
        print(f'[skip] {dest_subdir} already present at {dest}')
        return dest
    os.makedirs(dest, exist_ok=True)
    print(f'Downloading {dataset_ref} -> {dest} ...')
    subprocess.run(['kaggle', 'datasets', 'download', '-d', dataset_ref, '-p', dest, '--unzip'], check=True)
    return dest

V4_DATA_BACKUP = f'{PROJECT_DIR}/v4_data_backup.tar.gz'
restored_from_backup = False
if os.path.exists(V4_DATA_BACKUP):
    print('Found existing v4 data backup on Drive -- restoring instead of re-downloading.')
    subprocess.run(['cp', V4_DATA_BACKUP, '/content/v4_data_backup.tar.gz'], check=True)
    subprocess.run(['tar', '-xzf', '/content/v4_data_backup.tar.gz', '-C', '/content'], check=True)
    restored_from_backup = True
    print('Restored.')

    # The backup was archived relative to DATA_DIR but extracts relative to /content -- one
    # directory level off. Symlink so DATA_DIR/<name> resolves to the real extracted files,
    # BEFORE kaggle_download's existence check runs, so it doesn't wrongly re-download.
    os.makedirs(DATA_DIR, exist_ok=True)
    for name in ['adarshsingh_adf', 'deepvoice', 'deepvoice_chunks']:
        real = f'/content/{name}'
        link = f'{DATA_DIR}/{name}'
        if os.path.exists(real) and not os.path.exists(link):
            os.symlink(real, link)
            print(f'Linked {link} -> {real}')

adf_dir = kaggle_download('adarshsingh0903/audio-deepfake-detection-dataset', 'adarshsingh_adf')
deepvoice_dir = kaggle_download('birdy654/deep-voice-deepfake-voice-recognition', 'deepvoice')
print('Done.')


## Upload realistic eval dataset

Manually pick the zipped `audio` / `audio_eval_dataset.zip` folder when prompted.

In [ ]:
from google.colab import files
import zipfile

print('Upload the zipped "audio" folder (voice_changing dataset).')
uploaded = files.upload()
zip_name = next(iter(uploaded))

eval_dir = '/content/real_eval_audio'
os.makedirs(eval_dir, exist_ok=True)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(eval_dir)
print('Extracted to', eval_dir)


In [ ]:
import glob

eval_files = (
    glob.glob(f'{eval_dir}/**/*.mp3', recursive=True) +
    glob.glob(f'{eval_dir}/**/*.m4a', recursive=True) +
    glob.glob(f'{eval_dir}/**/*.wav', recursive=True)
)
print(f'Found {len(eval_files)} real-world eval files.')
assert eval_files, 'No audio files found in the extracted zip -- check its folder structure.'


In [ ]:
from pathlib import Path

HELD_OUT_SPEAKER = ('USA', 'male')  # never used in training -- pure held-out eval target

realistic_finetune_rows = []
realistic_heldout_rows = []
for path in eval_files:
    parts = Path(path).parts
    country, gender = parts[-4], parts[-3]
    stem = Path(path).stem
    label = 'genuine' if stem == 'original' else 'deepfake'
    row = {'path': path, 'label': label, 'source': 'realistic_voice_changing',
           'speaker_id': f'{country}_{gender}', 'split_hint': None}
    if (country, gender) == HELD_OUT_SPEAKER:
        realistic_heldout_rows.append(row)
    else:
        realistic_finetune_rows.append(row)

print(f'Fine-tune rows (3 speakers): {len(realistic_finetune_rows)}')
print(f'Held-out eval rows (speaker {HELD_OUT_SPEAKER}): {len(realistic_heldout_rows)}')
assert realistic_finetune_rows and realistic_heldout_rows, 'Expected data on both sides of the speaker split.'


## Helpers, model class, and the already-trained v2 model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from transformers import Wav2Vec2Model
import soundfile as sf
import librosa
import numpy as np
import pandas as pd
import glob
from pathlib import Path

EVAL_LABELS = ['genuine', 'ai_generated', 'deepfake']
TARGET_SR = 16000
WINDOW_SAMPLES = TARGET_SR * 4

def load_mono_16k(path):
    # soundfile can't open some containers (notably .m4a) -- fall back to librosa
    # (ffmpeg-backed) rather than silently failing.
    try:
        audio, sr = sf.read(path, always_2d=False)
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sr != TARGET_SR:
            audio = librosa.resample(audio.astype(np.float32), orig_sr=sr, target_sr=TARGET_SR)
        return audio.astype(np.float32)
    except Exception:
        audio, sr = librosa.load(path, sr=TARGET_SR, mono=True)
        return audio.astype(np.float32)

def rms_normalize(audio, target_dbfs=-20.0):
    rms = np.sqrt(np.mean(audio ** 2)) + 1e-9
    target_rms = 10 ** (target_dbfs / 20)
    return audio * (target_rms / rms)

def fixed_window(audio, length, train=False):
    if len(audio) >= length:
        if train:
            start = np.random.randint(0, len(audio) - length + 1)
        else:
            start = (len(audio) - length) // 2
        return audio[start:start + length]
    pad = length - len(audio)
    left = pad // 2
    right = pad - left
    return np.pad(audio, (left, right), mode='reflect')

class AudioDataset(Dataset):
    def __init__(self, manifest_df, split, train):
        self.rows = manifest_df[manifest_df['split'] == split].reset_index(drop=True)
        self.train = train

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        try:
            audio = load_mono_16k(row['path'])
            audio = rms_normalize(audio)
            audio = fixed_window(audio, WINDOW_SAMPLES, self.train)
        except Exception:
            audio = np.zeros(WINDOW_SAMPLES, dtype=np.float32)
        return torch.from_numpy(audio), LABEL_TO_IDX[row['label']]

class AudioClassifier(nn.Module):
    def __init__(self, num_classes=3, freeze_feature_extractor=True):
        super().__init__()
        self.backbone = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base')
        if freeze_feature_extractor:
            self.backbone.feature_extractor._freeze_parameters()
        hidden = self.backbone.config.hidden_size
        self.head = nn.Sequential(nn.Linear(hidden, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, num_classes))

    def forward(self, waveform):
        out = self.backbone(waveform).last_hidden_state
        return self.head(out.mean(dim=1))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    print('WARNING: no GPU detected. Runtime -> Change runtime type -> GPU, then re-run from here.')

eval_model = AudioClassifier(num_classes=3).to(device)
eval_model.load_state_dict(torch.load(f'{export_dir}/wav2vec2_deepfake_classifier.pt', map_location=device))
eval_model.eval()
print('Loaded v2 model on', device)


In [ ]:
ft_model = AudioClassifier(num_classes=len(LABELS)).to(device)
ft_model.load_state_dict(torch.load(f'{export_dir_v3}/wav2vec2_deepfake_classifier.pt', map_location=device))
ft_model.eval()
print('Loaded v3 model from', export_dir_v3)


In [ ]:
def infer_with(m, path):
    audio = load_mono_16k(path)
    audio = rms_normalize(audio)
    audio = fixed_window(audio, WINDOW_SAMPLES)
    waveform = torch.from_numpy(audio).unsqueeze(0).to(device)
    m.eval()
    with torch.no_grad():
        probs = F.softmax(m(waveform), dim=-1).cpu().numpy()[0]
    return probs


## Load the already-built v4 manifest (from the earlier combine step)

In [ ]:
manifest_path_v4 = f'{MANIFEST_DIR_V4}/v4_manifest.csv'
assert os.path.exists(manifest_path_v4), (
    f'{manifest_path_v4} not found -- the v4 manifest from the earlier combine step is '
    'missing. Stop here and flag this rather than rebuilding blind -- the rebuild needs '
    'several extra cells not included in this trimmed notebook.'
)
df_v4 = pd.read_csv(manifest_path_v4)
print('Loaded v4 manifest:', manifest_path_v4)
print(df_v4.groupby(['split', 'label']).size())


## Build v4's datasets and load its already-trained checkpoint

In [ ]:
train_ds_v4 = AudioDataset(df_v4, 'train', train=True)
val_ds_v4 = AudioDataset(df_v4, 'val', train=False)
test_ds_v4 = AudioDataset(df_v4, 'test', train=False)
print(f'train={len(train_ds_v4)}  val={len(val_ds_v4)}  test={len(test_ds_v4)}')

v4_model = AudioClassifier(num_classes=len(LABELS)).to(device)
v4_model.load_state_dict(torch.load(f'{CKPT_DIR_V4}/best.pt', map_location=device))
v4_model.eval()
print('Loaded v4 best checkpoint (already trained in an earlier session).')


## Train v5 -- same v4 data, gentle fine-tune from v2 (not a fresh retrain)

v4 (fresh wav2vec2-base, full retrain, lr=3e-5) collapsed to predicting "ai_generated" for
almost every held-out file despite good val F1 -- val F1 only reflects v2's original
distribution, never the held-out speaker, so it couldn't catch this. v3 (gentle low-lr
fine-tune from v2's weights) generalized fine on the same held-out speaker (45%). v5 tests
whether the collapse was specifically caused by training from scratch at a higher lr, by
reusing v4's exact same (already-balanced) manifest but fine-tuning from v2's weights at
v3's low lr instead -- one variable changed, not two.

In [ ]:
RUN_NAME_V5 = 'v5'
CKPT_DIR_V5 = f'{PROJECT_DIR}/checkpoints_{RUN_NAME_V5}'
LOG_DIR_V5 = f'{PROJECT_DIR}/logs_{RUN_NAME_V5}'
os.makedirs(CKPT_DIR_V5, exist_ok=True)
os.makedirs(LOG_DIR_V5, exist_ok=True)

v5_model = AudioClassifier(num_classes=len(LABELS)).to(device)
v5_model.load_state_dict(torch.load(f'{export_dir}/wav2vec2_deepfake_classifier.pt', map_location=device))
print('Loaded v2 weights as the v5 fine-tuning starting point.')


In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
from tqdm import tqdm
import csv, os, shutil

V5_BATCH_SIZE = 8
V5_GRAD_ACCUM = 4
V5_EPOCHS = 5       # a bit more than v3's 3 -- there's more new data mixed in this time
V5_LR = 1e-5        # v3's gentle rate, not v4's 3e-5 from-scratch rate

def collate(batch):
    waveforms = torch.stack([b[0] for b in batch])
    labels = torch.tensor([b[1] for b in batch])
    return waveforms, labels

v5_train_loader = DataLoader(train_ds_v4, batch_size=V5_BATCH_SIZE, shuffle=True, collate_fn=collate, num_workers=2)
v5_val_loader = DataLoader(val_ds_v4, batch_size=V5_BATCH_SIZE, shuffle=False, collate_fn=collate, num_workers=2)

train_counts = df_v4[df_v4['split'] == 'train']['label'].value_counts()
weights = torch.tensor([1.0 / train_counts.get(l, 1) for l in LABELS], dtype=torch.float32)
weights = weights / weights.sum() * len(LABELS)
weights = weights.to(device)
print('Class weights:', dict(zip(LABELS, weights.tolist())))

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = AdamW(v5_model.parameters(), lr=V5_LR)
total_steps = (len(v5_train_loader) // V5_GRAD_ACCUM) * V5_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.05 * total_steps), total_steps)

def evaluate_v5(loader):
    v5_model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for waveforms, labels in loader:
            waveforms = waveforms.to(device)
            logits = v5_model(waveforms)
            all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
            all_labels.extend(labels.numpy())
    return f1_score(all_labels, all_preds, average='macro')

LOCAL_CKPT_DIR_V5 = '/content/checkpoints_v5_local'
os.makedirs(LOCAL_CKPT_DIR_V5, exist_ok=True)

def save_ckpt_verified_v5(obj, filename):
    local_path = f'{LOCAL_CKPT_DIR_V5}/{filename}'
    drive_path = f'{CKPT_DIR_V5}/{filename}'
    torch.save(obj, local_path)
    local_size = os.path.getsize(local_path)
    shutil.copy2(local_path, drive_path)
    drive_ok = os.path.exists(drive_path) and os.path.getsize(drive_path) == local_size
    print(f'  [ckpt] {filename}: local={local_size/1e6:.1f}MB  drive_saved={drive_ok}')
    if not drive_ok:
        print(f'  [ckpt] WARNING: Drive copy failed for {filename} -- local fallback at {local_path}')
    return drive_ok

ckpt_path_v5 = f'{CKPT_DIR_V5}/latest.pt'
log_path_v5 = f'{LOG_DIR_V5}/training_log.csv'
start_epoch = 0
best_val_f1 = -1
epochs_no_improve = 0
PATIENCE = 2

pre_ft_val_f1 = evaluate_v5(v5_val_loader)
best_val_f1 = pre_ft_val_f1
print(f'val macro-F1 BEFORE v5 fine-tuning: {pre_ft_val_f1:.4f} (should roughly match v2\'s own val F1)')

if os.path.exists(ckpt_path_v5):
    state = torch.load(ckpt_path_v5, map_location=device)
    v5_model.load_state_dict(state['model'])
    optimizer.load_state_dict(state['optimizer'])
    start_epoch = state['epoch'] + 1
    best_val_f1 = state.get('best_val_f1', pre_ft_val_f1)
    epochs_no_improve = state.get('epochs_no_improve', 0)
    print(f'Resuming v5 training from epoch {start_epoch} (best so far: {best_val_f1:.4f})')
else:
    print('No checkpoint found -- starting v5 fresh from epoch 0.')

if not os.path.exists(log_path_v5):
    with open(log_path_v5, 'w', newline='') as f:
        csv.writer(f).writerow(['epoch', 'train_loss', 'val_macro_f1'])

for epoch in range(start_epoch, V5_EPOCHS):
    v5_model.train()
    running_loss = 0.0
    optimizer.zero_grad()
    for step, (waveforms, labels) in enumerate(tqdm(v5_train_loader, desc=f'v5 epoch {epoch}')):
        waveforms, labels = waveforms.to(device), labels.to(device)
        logits = v5_model(waveforms)
        loss = criterion(logits, labels) / V5_GRAD_ACCUM
        loss.backward()
        running_loss += loss.item() * V5_GRAD_ACCUM
        if (step + 1) % V5_GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(v5_model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    train_loss = running_loss / len(v5_train_loader)
    val_f1 = evaluate_v5(v5_val_loader)
    print(f'epoch {epoch}: train_loss={train_loss:.4f}  val_macro_f1={val_f1:.4f}')
    with open(log_path_v5, 'a', newline='') as f:
        csv.writer(f).writerow([epoch, train_loss, val_f1])

    improved = val_f1 > best_val_f1
    if improved:
        best_val_f1 = val_f1
        epochs_no_improve = 0
        save_ckpt_verified_v5(v5_model.state_dict(), 'best.pt')
    else:
        epochs_no_improve += 1

    save_ckpt_verified_v5({
        'model': v5_model.state_dict(), 'optimizer': optimizer.state_dict(),
        'epoch': epoch, 'best_val_f1': best_val_f1, 'epochs_no_improve': epochs_no_improve,
    }, 'latest.pt')

    if epochs_no_improve >= PATIENCE:
        print(f'Early stopping at epoch {epoch}.')
        break

print(f'\nv5 training done. val macro-F1: before={pre_ft_val_f1:.4f}, best after={best_val_f1:.4f}')
print(f'Local fallback dir (this session only): {LOCAL_CKPT_DIR_V5}')


## Verify the v5 checkpoint landed on Drive before evaluating

In [ ]:
print(sorted(os.listdir(CKPT_DIR_V5)))


## Evaluate v5 against the held-out speaker (v2 vs v3 vs v4 vs v5 -- the real test)

In [ ]:
best_state_v5 = torch.load(f'{CKPT_DIR_V5}/best.pt', map_location=device)
v5_model.load_state_dict(best_state_v5)
v5_model.eval()

rows = []
for r in realistic_heldout_rows:
    true_label = r['label']
    v2_probs = infer_with(eval_model, r['path'])
    v3_probs = infer_with(ft_model, r['path'])
    v4_probs = infer_with(v4_model, r['path'])
    v5_probs = infer_with(v5_model, r['path'])
    rows.append({
        'file': Path(r['path']).parent.name + '/' + Path(r['path']).stem,
        'true_label': true_label,
        'v2_pred': EVAL_LABELS[v2_probs.argmax()],
        'v3_pred': EVAL_LABELS[v3_probs.argmax()],
        'v4_pred': EVAL_LABELS[v4_probs.argmax()],
        'v5_pred': EVAL_LABELS[v5_probs.argmax()],
        'v5_genuine_p': v5_probs[0], 'v5_ai_generated_p': v5_probs[1], 'v5_deepfake_p': v5_probs[2],
    })

comparison_v5_df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', None)
print(comparison_v5_df.to_string(index=False))

for col in ['v2_pred', 'v3_pred', 'v4_pred', 'v5_pred']:
    acc = (comparison_v5_df['true_label'] == comparison_v5_df[col]).mean()
    print(f'{col} accuracy on held-out speaker: {acc:.1%}')

comparison_v5_df.to_csv(f'{PROJECT_DIR}/v2_v3_v4_v5_heldout_comparison.csv', index=False)


## Export v5

In [ ]:
export_dir_v5 = f'{PROJECT_DIR}/export_{RUN_NAME_V5}'
os.makedirs(export_dir_v5, exist_ok=True)
torch.save(v5_model.state_dict(), f'{export_dir_v5}/wav2vec2_deepfake_classifier.pt')
import json as json_lib
with open(f'{export_dir_v5}/label_map.json', 'w') as f:
    json_lib.dump({'labels': LABELS, 'model_name': 'facebook/wav2vec2-base',
                   'note': 'v5 = v2 fine-tuned (lr=1e-5) on v4\'s combined manifest '
                           '(v2 base + adarshsingh_adf + DEEP-VOICE + oversampled realistic '
                           'voice-changing data), not a from-scratch retrain'},
                  f, indent=2)
print('Exported v5 to', export_dir_v5)
